In [1]:
import numpy as np
from numpy import pi, arccos, cos, sin, tan, mean
import matplotlib.pyplot as plt
from cats.cdataframe import CDataFrame
import glob
import pandas as pd

Welcome to JupyROOT 6.28/10


#### Save physical constants which will be needed for later calculations

In [2]:
h     = 4.13566e-12 # MeV*ns
hbar  = h / (2 * np.pi) # MeV*ns
mass = {'h': 0.50 * 0.51099906 / (2.99792458 * 1e2)**2,
        'e': 1.4672e-6} # MeV ns^2 / mm^2
vsound = 0.009 # mm/ns # Longitudinal sound speed

#### Make dictionary of DMC and Debug filenames

In [13]:
Vs = ['0V','4V']#, '4V', '100V']
charges = ['e','h']

#DMC_filenames = {V: {charge: sorted(glob.glob(f'samples/{charge}_{V}_new/stepcounter_512408??_00000?.root')) + 
#                    sorted(glob.glob(f'samples/{charge}_{V}_new/stepcounter_512408??_0000??.root')) for charge in charges} for V in Vs}
DMC_filenames = {V: {charge: sorted(glob.glob(f'samples/{charge}_{V}_new/stepcounter_512408??_00000?.root')) for charge in charges} for V in Vs}

#Debug_filenames = {V: {charge: sorted(glob.glob(f'samples/{charge}_{V}_new/LukePhononEnergies_?')) + 
#                    sorted(glob.glob(f'samples/{charge}_{V}_new/LukePhononEnergies_??')) for charge in charges} for V in Vs}
Debug_filenames = {V: {charge: sorted(glob.glob(f'samples/{charge}_{V}_new/LukePhononEnergies_?')) for charge in charges} for V in Vs}

#### Extract DMC data and define a few cuts

In [14]:
parameters = (['EventNum', 'KE', 'KE3', 'Process', 'Charge', 'PName', 'TrkStep', 'Parent'] + 
              [label + sequence for sequence in ['1', '3'] for label in ['X', 'Y', 'Z', 'Xmom', 'Ymom', 'Zmom', 'Time', 'V']])

data = {V: {charge: None for charge in charges} for V in Vs}

for V in Vs:
    for charge in charges:
        mcHitCounter = CDataFrame('G4SimDir/mcHitCounter', DMC_filenames[V][charge])
        data[V][charge] = mcHitCounter.AsNumpy(parameters)
        eCut = (data[V][charge]['Charge'] == -1) & (data[V][charge]['Process'] == 'G4CMPLukeScattering')
        hCut = (data[V][charge]['Charge'] == 1) & (data[V][charge]['Process'] == 'G4CMPLukeScattering')
        phCut = (data[V][charge]['Charge'] == 0) & (data[V][charge]['TrkStep']%100000==1)
        data[V][charge]['hCut'] = hCut
        data[V][charge]['eCut'] = eCut
        data[V][charge]['phCut'] = phCut

In [ ]:
nCDataFrame('G4SettingsInfoDir/Geometry', DMC_filenames['4V']['h']).AsNumpy(['Voltage'])['Voltage']
data['4V']['h']

#### Match charges and emitted phonons. Now live under 'charge' and 'phonon' keys in the dataLuke dictionary.

In [5]:
def lukeSort(data):

    parameters = (['EventNum', 'KE', 'KE3', 'Process', 'Charge', 'PName', 'TrkStep', 'Parent'] + 
                  [label + sequence for sequence in ['1', '3'] for label in ['X', 'Y', 'Z', 'Xmom', 'Ymom', 'Zmom', 'Time', 'V']])

    dataLuke = {V: {charge: {'charge': {parameter:[] for parameter in parameters}, 
                             'phonon': {parameter:[] for parameter in parameters}} for charge in charges} for V in Vs}

    for V in Vs:
        print(V)
        for charge in charges:
            print(charge)
    
            for event in np.unique(data[V][charge]['EventNum']):
                print(event)
                eventCut = data[V][charge]['EventNum'] == event
                chargeCut = data[V][charge][charge+'Cut']
                phononCut = data[V][charge]['phCut']

                for i in range(len(data[V][charge]['X1'][chargeCut & eventCut])):
                    ParentID_Ph = data[V][charge]['Parent']
                    TrackID_h = data[V][charge]['TrkStep'][chargeCut & eventCut][i] // 100000
                    emissionCut = abs(data[V][charge]['Time1'] - data[V][charge]['Time3'][chargeCut & eventCut][i]) < 1e-9
                    lukeCut = TrackID_h == ParentID_Ph

                    for parameter in parameters:
                        dataLuke[V][charge]['charge'][parameter].append(data[V][charge][parameter][chargeCut & eventCut][i])
                    if any(emissionCut & lukeCut & phononCut):
                        for parameter in parameters:
                            dataLuke[V][charge]['phonon'][parameter].append(data[V][charge][parameter][lukeCut & emissionCut & phononCut & eventCut][0])

            for parameter in parameters:
                dataLuke[V][charge]['charge'][parameter] = np.array(dataLuke[V][charge]['charge'][parameter])
                dataLuke[V][charge]['phonon'][parameter] = np.array(dataLuke[V][charge]['phonon'][parameter])       

    return dataLuke

#### Add debug information that is not usually available in supersim output. Only interested in wavevectors and phonon scattering angle at the moment.

In [6]:
def addDebug(dataLuke):
    
    keys = ['Track ID', ' Track Type', 'Track Weight', 'Track Energy [eV]', 'Track Momentum [eV]', 'WaveVector', 'Phonon Theta', 
        'Phonon Energy [eV]', 'Phonon Weight', 'Recoil WaveVector', 'Final Energy [eV]', 'Final Momentum [eV]']
    dataDebug = {V: {charge: {} for charge in charges} for V in Vs}
    
    for V in Vs:
        for charge in charges:
            for key in ['WaveVector', 'Phonon Theta', 'Recoil WaveVector', 'EventNum'] + [label + sequence for sequence in ['1', '3'] for label in ['X', 'Y', 'Z', 'Time']]:
                dataDebug[V][charge].update({key: []})

            for file in range(len(Debug_filenames[V][charge])):
                eventCut = dataLuke[V][charge]['charge']['EventNum'] == file
                openFile = open(Debug_filenames[V][charge][file],'r')
                content = openFile.read().split('\n1')

                for row in range(1, len(content)):
                    dataDebug[V][charge]['EventNum'].append(file)
                    for key in [label + sequence for sequence in ['1', '3'] for label in ['X', 'Y', 'Z', 'Time']]:
                        dataDebug[V][charge][key].append(dataLuke[V][charge]['charge'][key][eventCut][row-1])
                    for column in range(len(content[row].split(','))):
                        if keys[column] in ['TrackID','WaveVector', 'Phonon Theta', 'Recoil WaveVector']:
                            dataDebug[V][charge][keys[column]].append(float(content[row].split(',')[column]))

                openFile.close()

            for i in list( dataDebug[V][charge].keys()):
                dataDebug[V][charge][i] = np.array(dataDebug[V][charge][i])

        return dataDebug

#### Calculate the magnitude of the phonon wavevector using

$q = 2 (|k| \cos(\theta) - k_s)$

##### Prerequisite: addDebug(dataLuke)

In [7]:
def calcq(dataDebug):
    for V in Vs:
        for charge in charges:
            k = dataDebug[V][charge]['WaveVector']
            k_recoil = dataDebug[V][charge]['Recoil WaveVector']
            theta = dataDebug[V][charge]['Phonon Theta']
            ksound = mass[charge] * vsound / hbar # mm^-1
            dataDebug[V][charge]['Phonon WaveVector'] = 2 * (k * np.cos(theta) - ksound)

    return dataDebug

#### Calculate the recoil scattering angle of the charge using

$\cos\theta_{kk'} = \frac{|k|}{2|k'|} + \frac{|k'|}{2|k|} - \frac{2}{|k| |k'|} (|k| \cos\theta - k_s)^2$

##### Prerequisite: addDebug(dataLuke), calcq(dataLuke)

In [8]:
def calcThetaKK(dataDebug):
    for V in Vs:
        for charge in charges:
            k =  dataDebug[V][charge]['WaveVector']
            k_recoil = dataDebug[V][charge]['Recoil WaveVector']
            theta = dataDebug[V][charge]['Phonon Theta']
            ksound = mass[charge] * vsound / hbar # mm^-1
            cos_theta = 0.5 * k / k_recoil + 0.5 * k_recoil / k - 2 / (k * k_recoil) * (k * np.cos(theta) - ksound)**2
            dataDebug[V][charge]['Recoil Theta'] = np.arccos(cos_theta)

    return dataDebug

In [9]:
dataLuke = lukeSort(data)

4V
e
0.0
1.0
2.0
3.0
4.0
5.0
6.0
7.0
8.0
9.0
h
0.0
1.0
2.0
3.0
4.0
5.0
6.0
7.0
8.0
9.0


In [10]:
dataDebug = addDebug(dataLuke)
dataDebug = calcq(dataDebug)
dataDebug = calcThetaKK(dataDebug)

In [11]:
pd.DataFrame(dataLuke['0V']['h']['charge']).to_csv("samples/h_0V_new/charge.csv", index=False)
pd.DataFrame(dataLuke['0V']['h']['phonon']).to_csv("samples/h_0V_new/phonon.csv", index=False)
pd.DataFrame(dataDebug['0V']['h']).to_csv("samples/h_0V_new/scatter.csv", index=False)

In [12]:
pd.DataFrame(dataLuke['0V']['e']['charge']).to_csv("samples/e_0V_new/charge.csv", index=False)
pd.DataFrame(dataLuke['0V']['e']['phonon']).to_csv("samples/e_0V_new/phonon.csv", index=False)
pd.DataFrame(dataDebug['0V']['e']).to_csv("samples/e_0V_new/scatter.csv", index=False)

In [11]:
pd.DataFrame(dataLuke['4V']['h']['charge']).to_csv("samples/h_4V_new/charge.csv", index=False)
pd.DataFrame(dataLuke['4V']['h']['phonon']).to_csv("samples/h_4V_new/phonon.csv", index=False)
pd.DataFrame(dataDebug['4V']['h']).to_csv("samples/h_4V_new/scatter.csv", index=False)

In [12]:
pd.DataFrame(dataLuke['4V']['e']['charge']).to_csv("samples/e_4V_new/charge.csv", index=False)
pd.DataFrame(dataLuke['4V']['e']['phonon']).to_csv("samples/e_4V_new/phonon.csv", index=False)
pd.DataFrame(dataDebug['4V']['e']).to_csv("samples/e_4V_new/scatter.csv", index=False)